# Plant Pathology 2021 Offline Inference Submission (v005)

This notebook is the offline submission-only companion to the training notebook.

- Keep Kaggle notebook Internet mode set to **Off**
- Attach the Plant Pathology 2021 competition dataset
- Ensure the v005 checkpoint path exists
- Run all cells to generate `submission.csv`

Inference policy in this version:
- full-image only (no tile inference)
- fixed resize to `518x518`
- fixed threshold `0.5` (no threshold tuning)



In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch import amp
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm

try:
    import timm
except ImportError as exc:
    raise ImportError(
        "`timm` is required for offline inference. "
        "Attach an offline dependency dataset if the Kaggle runtime does not include timm."
    ) from exc

BASE_DIR = "/kaggle/input/competitions/plant-pathology-2021-fgvc8"
CHECKPOINT_PATH = "/kaggle/input/models/junyaoshi6416/v005pth/transformers/default/1/v005pth.pth"
MODEL_NAME = "vit_small_patch14_dinov2.lvd142m"
IMAGE_SIZE = 518
EVAL_BATCH_SIZE = 32
DEFAULT_NUM_WORKERS = 2
THRESHOLD = 0.5

ALL_LABELS = [
    "complex",
    "frog_eye_leaf_spot",
    "healthy",
    "powdery_mildew",
    "rust",
    "scab",
]
NUM_CLASSES = len(ALL_LABELS)

TEST_DIR = os.path.join(BASE_DIR, "test_images")
SAMPLE_SUBMISSION_PATH = os.path.join(BASE_DIR, "sample_submission.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NUM_WORKERS = DEFAULT_NUM_WORKERS if DEVICE.type == "cuda" else 0
PIN_MEMORY = DEVICE.type == "cuda"
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm:", timm.__version__)
print("Device:", DEVICE)
print("Checkpoint path:", CHECKPOINT_PATH)
print("Inference policy: full-image 518, threshold=0.5")


In [ ]:
if not Path(BASE_DIR).exists():
    raise FileNotFoundError(
        f"Competition dataset not found at {BASE_DIR}. Attach Plant Pathology 2021 dataset first."
    )

if not Path(TEST_DIR).exists():
    raise FileNotFoundError(f"Test image directory not found: {TEST_DIR}")

if not Path(SAMPLE_SUBMISSION_PATH).exists():
    raise FileNotFoundError(f"sample_submission.csv not found: {SAMPLE_SUBMISSION_PATH}")

if not Path(CHECKPOINT_PATH).exists():
    raise FileNotFoundError(
        f"Checkpoint not found at {CHECKPOINT_PATH}. Confirm the v005 model dataset is attached."
    )

sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
test_image_names = sample_submission["image"].tolist()

print("Sample submission shape:", sample_submission.shape)
print("Test image count:", len(test_image_names))
print(sample_submission.head())


In [ ]:
class PlantPathologyTestDataset(Dataset):
    def __init__(self, image_names, image_dir, transform=None):
        self.image_names = list(image_names)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]
        image_path = os.path.join(self.image_dir, image_name)
        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image_tensor = self.transform(image) if self.transform else image
        return image_tensor, image_name


def build_dataloader_kwargs(batch_size):
    kwargs = {
        "batch_size": batch_size,
        "shuffle": False,
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY,
    }
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = PERSISTENT_WORKERS
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return kwargs


probe_model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=0)
DATA_CONFIG = timm.data.resolve_model_data_config(probe_model)
del probe_model

NORMALIZE_MEAN = DATA_CONFIG.get("mean", (0.485, 0.456, 0.406))
NORMALIZE_STD = DATA_CONFIG.get("std", (0.229, 0.224, 0.225))

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])

test_dataset = PlantPathologyTestDataset(
    image_names=test_image_names,
    image_dir=TEST_DIR,
    transform=eval_transform,
)

test_loader = DataLoader(
    test_dataset,
    **build_dataloader_kwargs(batch_size=EVAL_BATCH_SIZE),
)

print("Normalization mean:", NORMALIZE_MEAN)
print("Normalization std:", NORMALIZE_STD)
print("Eval batch size:", EVAL_BATCH_SIZE)
print("DataLoader workers:", NUM_WORKERS)


In [ ]:
class PlantDiseaseDINOv2Model(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0)
        self.feature_dim = getattr(self.backbone, "num_features", None)
        if self.feature_dim is None:
            raise ValueError("Could not infer backbone feature dimension.")

        self.head = nn.Sequential(
            nn.LayerNorm(self.feature_dim),
            nn.Dropout(0.3),
            nn.Linear(self.feature_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)

        if isinstance(features, dict):
            features = next(iter(features.values()))
        elif isinstance(features, (list, tuple)):
            features = features[0]

        if features.ndim == 3:
            features = features[:, 0]

        return self.head(features)


def extract_state_dict(checkpoint_obj):
    if isinstance(checkpoint_obj, dict) and "model_state" in checkpoint_obj:
        state_dict = checkpoint_obj["model_state"]
        source = "checkpoint['model_state']"
    elif isinstance(checkpoint_obj, dict):
        state_dict = checkpoint_obj
        source = "checkpoint_as_state_dict"
    else:
        raise TypeError(
            f"Unsupported checkpoint type: {type(checkpoint_obj)}. Expected a dict-like object."
        )

    if not isinstance(state_dict, dict):
        raise TypeError("Resolved state_dict is not a dict.")

    return state_dict, source


def load_checkpoint_object(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=True), "weights_only=True"
    except TypeError:
        return torch.load(path, map_location=device), "legacy_torch_load"


def evaluate_image_level(model, loader, device):
    model.eval()
    all_probabilities = []
    all_names = []

    with torch.no_grad():
        progress = tqdm(loader, desc="Test inference", leave=False)
        for images, image_names in progress:
            images = images.to(device, non_blocking=True)
            with amp.autocast(device_type=device.type, enabled=USE_AMP):
                logits = model(images)

            probs = torch.sigmoid(logits).float().cpu().numpy()
            all_probabilities.append(probs)
            all_names.extend(list(image_names))

    return {
        "probs": np.concatenate(all_probabilities, axis=0),
        "image_names": all_names,
    }


def probs_to_label(prob_row, threshold=THRESHOLD):
    selected = [ALL_LABELS[i] for i, prob in enumerate(prob_row) if prob >= threshold]
    if not selected:
        selected = [ALL_LABELS[int(np.argmax(prob_row))]]
    return " ".join(selected)


submission_model = PlantDiseaseDINOv2Model(MODEL_NAME, NUM_CLASSES).to(DEVICE)

try:
    checkpoint_obj, load_mode = load_checkpoint_object(CHECKPOINT_PATH, DEVICE)
except Exception as exc:
    raise RuntimeError(f"Failed to load checkpoint from {CHECKPOINT_PATH}") from exc

state_dict, state_source = extract_state_dict(checkpoint_obj)

try:
    submission_model.load_state_dict(state_dict)
except RuntimeError as exc:
    raise RuntimeError(
        "Checkpoint weights do not match PlantDiseaseDINOv2Model. "
        "Confirm MODEL_NAME and classifier head match the training notebook."
    ) from exc

submission_model.eval()
checkpoint_stage = checkpoint_obj.get("stage", "unknown") if isinstance(checkpoint_obj, dict) else "unknown"

print("Checkpoint loaded with:", load_mode)
print("State dict source:", state_source)
if state_source == "checkpoint_as_state_dict":
    print("Fallback mode used: checkpoint has no 'model_state'; treated checkpoint as state_dict.")
print("Checkpoint stage:", checkpoint_stage)


In [ ]:
test_outputs = evaluate_image_level(
    model=submission_model,
    loader=test_loader,
    device=DEVICE,
)

test_probabilities = test_outputs["probs"]
predicted_image_names = test_outputs["image_names"]

if len(predicted_image_names) != len(test_image_names):
    raise ValueError(
        f"Predicted image count {len(predicted_image_names)} does not match expected {len(test_image_names)}."
    )

predicted_labels = [probs_to_label(prob_row, threshold=THRESHOLD) for prob_row in test_probabilities]

submission = pd.DataFrame({
    "image": predicted_image_names,
    "labels": predicted_labels,
})

if submission["image"].duplicated().any():
    duplicated = submission.loc[submission["image"].duplicated(), "image"].head().tolist()
    raise ValueError(f"Duplicate predicted image names detected, examples: {duplicated}")

submission = submission.set_index("image").reindex(sample_submission["image"]).reset_index()

if submission["labels"].isna().any():
    missing_rows = submission.loc[submission["labels"].isna(), "image"].head().tolist()
    raise ValueError(f"Missing predictions for images, examples: {missing_rows}")

submission.to_csv("submission.csv", index=False)

if len(submission) != len(sample_submission):
    raise ValueError(
        f"Submission row count {len(submission)} does not match sample submission row count {len(sample_submission)}."
    )

missing_columns = set(sample_submission.columns) - set(submission.columns)
if missing_columns:
    raise ValueError(f"Submission is missing expected columns: {sorted(missing_columns)}")

empty_rows = int((submission["labels"].str.len() == 0).sum())

print("submission.csv saved!")
print("\nFirst 5 rows:")
print(submission.head())
print("\nShape:", submission.shape)
print("Rows with empty labels:", empty_rows)
